# Supply Chain Assistant: Stress-Testing `OracleSemanticCache` and `OracleChatMessageHistory`

This notebook builds a **LangChain-powered supply chain assistant** whose job is
to answer operational questions (inventory, shipments, suppliers, purchase
orders) for a fictional distribution company. The assistant is deliberately
kept simple so that the focus stays on the two Oracle Database integrations
we want to exercise end to end:

1. **`OracleSemanticCache`** — semantic (vector-similarity) cache for LLM
   responses, backed by Oracle AI Vector Search. Paraphrased prompts can be
   served from cache, cutting latency and token spend.
2. **`OracleChatMessageHistory`** — durable, session-scoped chat history
   stored as rows in an Oracle table. Survives process restarts, supports
   many concurrent sessions, and exposes a bounded-history mode.

### Who this notebook is for

You are an **AI developer** building agentic systems on Oracle Database. You
already know LangChain basics (`@tool`, chat models, agents) and want a clear,
reproducible way to validate that the `langchain-oracledb` cache and history
components behave correctly under real workloads before you put them in front
of users.

### What you will do

- Wire up an Oracle connection, an embedding model, and a chat LLM.
- Exercise `OracleSemanticCache` across **five stress tests** that probe
  paraphrase recall, the score threshold, LLM-string isolation, latency, and
  scoped deletes.
- Exercise `OracleChatMessageHistory` across **four stress tests** that probe
  multi-session isolation, the bounded-history window, concurrent writes, and
  full-session replacement.
- Assemble a supply chain ReAct agent with domain tools, then run a
  **combined stress test** where both the cache and the history are exercised
  together by a multi-turn, paraphrase-heavy conversation.

> **Heads up.** Every code cell is meant to be read and run top to bottom. The
> stress tests are deliberately noisy — they print timings, hit/miss counts,
> and row-level diagnostics — because that is the point. Use those numbers as
> a baseline when you tune this for your own workload.


## 1. Prerequisites

You need:

| Requirement | Notes |
|---|---|
| Oracle Database 23ai (or Autonomous Database) | Vector search and JSON columns are required. The free `oracle/database-free` container image works. |
| Python 3.10+ | Matches the `langchain-oracledb` support matrix. |
| An OpenAI API key | This notebook uses `ChatOpenAI` via `langchain-openai` for the agent. Any LangChain `BaseChatModel` will work — swap the model in one cell. |
| An embedding model | Used by `OracleSemanticCache` to vectorize prompts. We default to a local `sentence-transformers` model so the notebook runs without external embedding calls. |

### Environment variables

The notebook reads these — set them before launching Jupyter (or use a `.env`
loader). None are written back. **The database defaults below match the ones
used by `examples/research_agent_with_oracle.ipynb` in the sibling
`langgraph-oracledb` project**, so one running Oracle container can serve both
notebooks without reconfiguration.

```bash
export DB_USER="VECTOR"
export DB_PASSWORD="VectorPwd_2025"
export DB_CONNECT_STRING="localhost:1521/FREEPDB1"

# Needed for the agent section (OpenAI)
export OPENAI_API_KEY="sk-..."
```


In [ ]:
# Install dependencies. Re-run this cell in a fresh kernel.
%pip install -qU \
    langchain-oracledb \
    langchain-core \
    langchain-community \
    langchain-openai \
    langchain-huggingface \
    langgraph \
    sentence-transformers \
    oracledb

import langchain_oracledb
print("langchain_oracledb:", langchain_oracledb.__file__)


## 2. Setup

We create three things up front and reuse them everywhere:

1. An **Oracle connection** (`oracledb.Connection`).
2. An **embedding model** for the semantic cache.
3. A **chat LLM** for the agent.

If you swap out the LLM or embeddings provider, these three cells are the only
places you need to touch.


In [2]:
import os
import time
import uuid
from pprint import pprint

import oracledb

# Credential defaults shared with `examples/research_agent_with_oracle.ipynb`
# in the sibling `langgraph-oracledb` project. `setdefault` only fills in a
# value if the env var is not already set — so a real export still wins.
os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_CONNECT_STRING = os.environ["DB_CONNECT_STRING"]

connection = oracledb.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    dsn=DB_CONNECT_STRING,
)
print(f"Connected to Oracle: version={connection.version}, dsn={DB_CONNECT_STRING}")


Connected to Oracle: version=23.26.1.0.0, dsn=localhost:1521/FREEPDB1


In [ ]:
import os
import getpass

# Prompt for the OpenAI API key if it isn't already in the environment.
# Using getpass avoids echoing the key to the terminal and keeps it out of
# the committed notebook output.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")


### 2.1 Embedding model

`OracleSemanticCache` needs an embedding model so it can project each prompt
into a vector for similarity search. We use a small local sentence-transformer
so that this notebook runs with no external dependencies — **do not use this
model for production caching**, swap it for the same embedder your retrieval
stack uses so prompts and cached entries share a vector space.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

# Quick sanity check — the vector should be 384-dim for MiniLM.
sample_vector = embeddings.embed_query("how many units of SKU-1001 are in stock?")
print(f"embedding dim = {len(sample_vector)}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9700.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dim = 384


### 2.2 Chat LLM

We use `ChatOpenAI` from `langchain-openai` because it's the shortest path to
a working agent — no cloud auth files, no compartment ids, just an API key.
If you prefer a different provider, replace this cell with `ChatAnthropic`,
`ChatOCIGenAI`, `ChatVertexAI`, etc. — the rest of the notebook is
provider-agnostic because it only uses `BaseChatModel` methods.


In [5]:
from langchain_openai import ChatOpenAI

# `ChatOpenAI` reads OPENAI_API_KEY from the environment by default.
assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY before running the agent section."
)

# Note: gpt-5.x models expect `max_completion_tokens` (the old `max_tokens`
# argument is silently ignored and the model may return an empty final
# response, which surfaces here as the agent terminating on a tool result
# instead of producing a summary).
llm = ChatOpenAI(
    model="gpt-5.2",
    temperature=0.2,
    max_completion_tokens=600,
)
print("LLM ready:", llm.model_name)


LLM ready: gpt-5.2


## 3. `OracleSemanticCache` — directed stress tests

Before we plug the cache into the agent, we drive it directly through its
public API. This isolates **cache behavior** from **LLM behavior** so that any
surprise we see has only one possible cause.

We will cover:

| # | Stress test | What it proves |
|---|---|---|
| 3.1 | Smoke test — exact hit / exact miss | Round-trip works and idempotent writes collapse to one row. |
| 3.2 | Paraphrase recall | Semantically equivalent questions reuse one cached answer. |
| 3.3 | Score threshold tuning | `score_threshold` blocks false-positive semantic hits. |
| 3.4 | `llm_string` isolation | Same prompt, different model config, different cached answer. |
| 3.5 | Bulk latency comparison | Cached lookup is dramatically cheaper than an LLM call. |
| 3.6 | Scoped `clear()` | `prompt=` and `llm_string=` filters remove only what they should. |


In [6]:
from langchain_oracledb import OracleSemanticCache
from langchain_core.outputs import Generation
from langchain_core.globals import set_llm_cache

CACHE_TABLE = f"SC_DEMO_CACHE_{uuid.uuid4().hex[:6].upper()}"

semantic_cache = OracleSemanticCache(
    client=connection,
    embedding=embeddings,
    table_name=CACHE_TABLE,
    # Threshold is a MAX cosine distance (0.0 = identical, ~1.0 = unrelated).
    # For `all-MiniLM-L6-v2`, natural-language paraphrases of a short query
    # typically land in the 0.25-0.45 range, so 0.35 is a reasonable default
    # to demonstrate paraphrase recall. Lower it for stricter matching.
    score_threshold=0.35,
)
print(f"Semantic cache table: {CACHE_TABLE}")


Semantic cache table: SC_DEMO_CACHE_C104DE


### 3.1 Smoke test — one write, two reads

We write a single response for `(prompt, llm_string)`, look it up with the
exact same arguments (expect a hit), then look up a clearly unrelated prompt
(expect a miss). This confirms the connection, the table, and the distance
function are all doing what we expect.


In [7]:
llm_key = "supply-chain-demo::gpt-5.2::t=0.2"

semantic_cache.update(
    "What is the current stock level of SKU-1001 in the Chicago warehouse?",
    llm_key,
    [Generation(text="SKU-1001 currently has 1,240 units on hand in Chicago-A.")],
)

hit = semantic_cache.lookup(
    "What is the current stock level of SKU-1001 in the Chicago warehouse?",
    llm_key,
)
miss = semantic_cache.lookup(
    "Tell me a bedtime story about a dragon.",
    llm_key,
)

print("exact hit   :", hit)
print("unrelated   :", miss)


exact hit   : [Generation(text='SKU-1001 currently has 1,240 units on hand in Chicago-A.')]
unrelated   : None


### 3.2 Paraphrase recall — the core value proposition

The whole point of a **semantic** cache is that two prompts that *mean the
same thing* should share one cached answer. We insert the answer once, then
hit it with nine paraphrases that no keyword cache would ever match. Every
paraphrase should come back with the same cached `Generation`.

> **Tuning note.** If a paraphrase misses, it's almost always because the
> distance exceeded `score_threshold`. Raise the threshold, or use a stronger
> embedding model — do *not* re-insert the paraphrase as a new cache entry,
> or you will fragment the cache and lose the win you came for.


In [8]:
paraphrases = [
    "How many units of SKU-1001 are in stock right now?",
    "Current inventory count for SKU-1001?",
    "Do we have SKU-1001 available? What is the quantity?",
    "Stock on hand for part number SKU-1001?",
    "Warehouse inventory for SKU-1001, please.",
    "How much SKU-1001 is left in Chicago?",
    "Whats the quantity of SKU-1001 in our Chicago DC?",
    "Give me the on-hand count for SKU-1001.",
    "SKU-1001 stock position?",
]

hits, misses = 0, 0
for p in paraphrases:
    result = semantic_cache.lookup(p, llm_key)
    if result is None:
        misses += 1
        print(f"MISS: {p!r}")
    else:
        hits += 1

print(f"\nParaphrase recall: {hits}/{len(paraphrases)} hits, {misses} misses")


MISS: 'Do we have SKU-1001 available? What is the quantity?'
MISS: 'Give me the on-hand count for SKU-1001.'

Paraphrase recall: 7/9 hits, 2 misses


### 3.3 Score threshold tuning

`score_threshold` is a **maximum allowed distance** (Oracle returns distance,
not similarity — lower is closer). We rebuild the cache with a very strict
threshold and confirm that even moderately rephrased prompts now miss. This
is the knob you will spend the most time tuning in production.


In [9]:
strict_cache = OracleSemanticCache(
    client=connection,
    embedding=embeddings,
    table_name=CACHE_TABLE,
    score_threshold=0.01,  # basically demand identical prompts
)

probe = "Stock on hand for part number SKU-1001?"
strict_result = strict_cache.lookup(probe, llm_key)
loose_result = semantic_cache.lookup(probe, llm_key)

print(f"strict threshold (0.01) -> {'HIT' if strict_result else 'MISS'}")
print(f"loose threshold  (0.35) -> {'HIT' if loose_result else 'MISS'}")


strict threshold (0.01) -> MISS
loose threshold  (0.35) -> HIT


### 3.4 `llm_string` isolation

The cache key is `(prompt, llm_string)`. Two different LLM configurations — a
different model, a different temperature, a different system prompt — must
not collide even when the user-visible prompt is identical. We verify that
by writing two entries for the same prompt under different `llm_string`s.


In [10]:
prompt = "Which supplier has the shortest lead time for bearings?"

semantic_cache.update(prompt, "model=gpt-5.2::t=0.2",
                     [Generation(text="Midwest Bearings Co. (SUP-301), 7-day lead time.")])
semantic_cache.update(prompt, "model=gpt-5.2-mini::t=0.7",
                     [Generation(text="Per my records, SUP-301 Midwest Bearings is fastest at a week.")])

print("full key:", semantic_cache.lookup(prompt, "model=gpt-5.2::t=0.2"))
print("mini key:", semantic_cache.lookup(prompt, "model=gpt-5.2-mini::t=0.7"))


full key: [Generation(text='Midwest Bearings Co. (SUP-301), 7-day lead time.')]
mini key: [Generation(text='Per my records, SUP-301 Midwest Bearings is fastest at a week.')]


### 3.5 Bulk latency comparison

Now we measure the thing that justifies having a cache at all: **lookup is
orders of magnitude cheaper than a model call**. We time one cold LLM call
against 100 cache lookups of varied paraphrases. In most setups the cached
path is **50×–500×** faster and costs nothing in tokens.


In [11]:
llm_prompt = "Give me a one-sentence status of SKU-1001 inventory."

t0 = time.perf_counter()
llm_response = llm.invoke(llm_prompt)
llm_latency_ms = (time.perf_counter() - t0) * 1000

# Cache that answer under our demo llm_key so the paraphrases resolve.
semantic_cache.update(llm_prompt, llm_key, [Generation(text=llm_response.content)])

bulk_prompts = paraphrases * 12  # ~108 lookups
t0 = time.perf_counter()
for p in bulk_prompts:
    semantic_cache.lookup(p, llm_key)
cache_total_ms = (time.perf_counter() - t0) * 1000

print(f"1 LLM call          : {llm_latency_ms:8.1f} ms")
print(f"{len(bulk_prompts):>3} cache lookups  : {cache_total_ms:8.1f} ms  "
      f"({cache_total_ms/len(bulk_prompts):.2f} ms avg)")
print(f"speedup per call    : {llm_latency_ms / (cache_total_ms/len(bulk_prompts)):.0f}x")


1 LLM call          :   1603.6 ms
108 cache lookups  :   1705.8 ms  (15.79 ms avg)
speedup per call    : 102x


### 3.6 Scoped `clear()` — deleting only what you meant to

`clear()` supports two filters: `prompt=` (exact prompt match) and
`llm_string=` (exact llm_string match). Anything else raises. We verify that
clearing by `llm_string` leaves other configurations' entries intact — this
is how you invalidate a model upgrade without nuking the whole cache.


In [12]:
before = {
    "full": semantic_cache.lookup("Which supplier has the shortest lead time for bearings?",
                                   "model=gpt-5.2::t=0.2"),
    "mini": semantic_cache.lookup("Which supplier has the shortest lead time for bearings?",
                                   "model=gpt-5.2-mini::t=0.7"),
}

semantic_cache.clear(llm_string="model=gpt-5.2-mini::t=0.7")

after = {
    "full": semantic_cache.lookup("Which supplier has the shortest lead time for bearings?",
                                   "model=gpt-5.2::t=0.2"),
    "mini": semantic_cache.lookup("Which supplier has the shortest lead time for bearings?",
                                   "model=gpt-5.2-mini::t=0.7"),
}

print("before clear:", {k: "present" if v else "gone" for k, v in before.items()})
print("after  clear:", {k: "present" if v else "gone" for k, v in after.items()})


before clear: {'full': 'present', 'mini': 'present'}
after  clear: {'full': 'present', 'mini': 'gone'}


## 4. `OracleChatMessageHistory` — directed stress tests

Chat history looks boring until you run a real service on top of it. Three
things usually break first: **session isolation** (messages leaking across
users), **unbounded context growth** (token costs exploding), and
**concurrent writes** (multiple turns landing at once from a noisy client).
We stress each of those.

| # | Stress test | What it proves |
|---|---|---|
| 4.1 | Basic round-trip | `add_messages` / `get_messages` / `clear` for one session. |
| 4.2 | Multi-session isolation | Many sessions in one table do not leak. |
| 4.3 | `history_size` window | Only the last N messages are returned, but older rows stay on disk. |
| 4.4 | Concurrent writes | Parallel `add_messages` calls preserve all rows and session scoping. |
| 4.5 | `messages` setter replacement | Full-session rewrite is atomic. |


In [13]:
from langchain_oracledb import OracleChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

HISTORY_TABLE = f"SC_DEMO_HISTORY_{uuid.uuid4().hex[:6].upper()}"

# One table, many sessions. Create it once here and reuse for the tests below.
OracleChatMessageHistory.create_tables(connection, HISTORY_TABLE)
print(f"Chat history table: {HISTORY_TABLE}")


Chat history table: SC_DEMO_HISTORY_E91047


### 4.1 Basic round-trip

A single session. We write three messages, read them back (order preserved),
then clear.


In [14]:
session_a = OracleChatMessageHistory(
    session_id="alice-2026-04-21",
    client=connection,
    table_name=HISTORY_TABLE,
    create_table=False,
    create_index=False,
)

session_a.add_messages([
    SystemMessage(content="You are a supply chain assistant."),
    HumanMessage(content="Hi, I'm Alice from logistics."),
    AIMessage(content="Hello Alice, how can I help?"),
])

for m in session_a.messages:
    print(f"{m.__class__.__name__:14} {m.content}")

session_a.clear()
print("after clear:", session_a.messages)


SystemMessage  You are a supply chain assistant.
HumanMessage   Hi, I'm Alice from logistics.
AIMessage      Hello Alice, how can I help?
after clear: []


### 4.2 Multi-session isolation

We interleave writes across ten sessions and confirm each session only sees
its own messages. This is the test to run first when you suspect a bug where
one user is seeing another user's history.


In [15]:
session_ids = [f"session-{i:02d}" for i in range(10)]
histories = {
    sid: OracleChatMessageHistory(
        session_id=sid,
        client=connection,
        table_name=HISTORY_TABLE,
        create_table=False,
        create_index=False,
    )
    for sid in session_ids
}

# Interleaved writes: round-robin across sessions, three turns each.
for turn in range(3):
    for sid in session_ids:
        histories[sid].add_messages([
            HumanMessage(content=f"{sid} question turn {turn}"),
            AIMessage(content=f"{sid} answer turn {turn}"),
        ])

# Each session should have exactly 6 messages, all tagged with its own id.
for sid in session_ids[:3]:
    msgs = histories[sid].messages
    print(f"{sid}: {len(msgs)} msgs, "
          f"all scoped = {all(sid in m.content for m in msgs)}")
print("... (remaining sessions look the same)")


session-00: 6 msgs, all scoped = True
session-01: 6 msgs, all scoped = True
session-02: 6 msgs, all scoped = True
... (remaining sessions look the same)


### 4.3 `history_size` — bounded context window

Large histories inflate token costs. `history_size=N` returns only the N most
recent messages at read time — the older rows stay on disk, so you can raise
or lower the window later without losing anything. We push 40 turns into a
session and confirm a `history_size=6` reader only sees the last six.


In [16]:
long_session = OracleChatMessageHistory(
    session_id="long-session-01",
    client=connection,
    table_name=HISTORY_TABLE,
    create_table=False,
    create_index=False,
)

for i in range(40):
    long_session.add_messages([
        HumanMessage(content=f"Q{i}: check SKU-{1000 + (i % 5)}"),
        AIMessage(content=f"A{i}: stock OK"),
    ])

bounded = OracleChatMessageHistory(
    session_id="long-session-01",
    client=connection,
    table_name=HISTORY_TABLE,
    create_table=False,
    create_index=False,
    history_size=6,
)

full = long_session.messages
window = bounded.messages
print(f"full session  : {len(full)} messages on disk")
print(f"history_size=6: {len(window)} messages returned")
print("last 6 from bounded reader:")
for m in window:
    print(f"  {m.__class__.__name__:14} {m.content}")


full session  : 80 messages on disk
history_size=6: 6 messages returned
last 6 from bounded reader:
  HumanMessage   Q37: check SKU-1002
  AIMessage      A37: stock OK
  HumanMessage   Q38: check SKU-1003
  AIMessage      A38: stock OK
  HumanMessage   Q39: check SKU-1004
  AIMessage      A39: stock OK


### 4.4 Concurrent writes

Two threads writing to the same session should not lose messages, and writes
to *different* sessions from the same pool should not leak. We spin up 20
threads split across two sessions and check the row counts match what we
sent.

> **About connections and threads.** `python-oracledb` `Connection` objects
> are not thread-safe for simultaneous use. In production you want a
> `ConnectionPool` and let each thread check out its own connection.
> `OracleChatMessageHistory` accepts either a `Connection` or a
> `ConnectionPool` — we use a pool here to make that pattern explicit.


In [17]:
import threading

pool = oracledb.create_pool(
    user=DB_USER,
    password=DB_PASSWORD,
    dsn=DB_CONNECT_STRING,
    min=2,
    max=8,
    increment=1,
)

def writer(session_id: str, n_messages: int, tag: str) -> None:
    hist = OracleChatMessageHistory(
        session_id=session_id,
        client=pool,
        table_name=HISTORY_TABLE,
        create_table=False,
        create_index=False,
    )
    for i in range(n_messages):
        hist.add_messages([HumanMessage(content=f"{tag}-msg-{i}")])

threads = []
for i in range(10):
    threads.append(threading.Thread(target=writer, args=("concurrent-A", 20, f"A{i}")))
    threads.append(threading.Thread(target=writer, args=("concurrent-B", 20, f"B{i}")))

for t in threads:
    t.start()
for t in threads:
    t.join()

read_a = OracleChatMessageHistory(
    session_id="concurrent-A", client=pool, table_name=HISTORY_TABLE,
    create_table=False, create_index=False,
)
read_b = OracleChatMessageHistory(
    session_id="concurrent-B", client=pool, table_name=HISTORY_TABLE,
    create_table=False, create_index=False,
)

print(f"session A: expected 200, got {len(read_a.messages)}")
print(f"session B: expected 200, got {len(read_b.messages)}")
print(f"A leaked into B? {any('A' in m.content for m in read_b.messages)}")
print(f"B leaked into A? {any('B' in m.content for m in read_a.messages)}")


session A: expected 200, got 200
session B: expected 200, got 200
A leaked into B? False
B leaked into A? False


### 4.5 Full-session replacement via the `messages` setter

Sometimes you need to **rewrite** a session — for example after summarizing
older turns into a single `SystemMessage` to shrink the context. The
`messages` setter is transactional: the delete + insert runs in one unit of
work and rolls back together on error.


In [18]:
rewrite_session = OracleChatMessageHistory(
    session_id="rewrite-session",
    client=connection,
    table_name=HISTORY_TABLE,
    create_table=False,
    create_index=False,
)

rewrite_session.add_messages([
    HumanMessage(content="Turn 1"),
    AIMessage(content="Response 1"),
    HumanMessage(content="Turn 2"),
    AIMessage(content="Response 2"),
])
print("before rewrite:", [m.content for m in rewrite_session.messages])

rewrite_session.messages = [
    SystemMessage(content="Summary: user asked 2 questions about turn 1 and turn 2."),
]
print("after rewrite :", [m.content for m in rewrite_session.messages])


before rewrite: ['Turn 1', 'Response 1', 'Turn 2', 'Response 2']
after rewrite : ['Summary: user asked 2 questions about turn 1 and turn 2.']


## 5. The supply chain assistant agent

Now we combine everything. We will build a ReAct agent with four tools that
return mock but realistic supply-chain data. Then we:

- Wire `OracleSemanticCache` to the LLM via `set_llm_cache` so that any repeat
  or paraphrased question short-circuits straight to a cached response.
- Wire `OracleChatMessageHistory` around each user turn so that the agent has
  durable, session-scoped memory across invocations.

The tools below return static data on purpose — a real deployment would hit
an ERP, a WMS, or a shipping provider API. Keeping the tools deterministic
means the cache hit/miss tests below are reproducible.


In [19]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

INVENTORY = {
    "SKU-1001": {"name": "Industrial Ball Bearing 6203", "stock": 1240,
                  "warehouse": "Chicago-A", "reorder_point": 500},
    "SKU-1002": {"name": "Stainless Steel Bolt M12x80",  "stock": 87,
                  "warehouse": "Dallas-C",  "reorder_point": 200},
    "SKU-1003": {"name": "Hydraulic Hose 3/8in",         "stock": 0,
                  "warehouse": "Dallas-C",  "reorder_point": 150},
    "SKU-1004": {"name": "Automotive Sensor Module v3",  "stock": 2150,
                  "warehouse": "Los Angeles-B", "reorder_point": 400},
    "SKU-1005": {"name": "Polymer Resin 25kg drum",      "stock": 340,
                  "warehouse": "Houston-A", "reorder_point": 100},
}

SUPPLIERS = {
    "SUP-301": {"name": "Midwest Bearings Co.", "lead_time_days": 7,
                 "region": "US-Midwest", "rating": 4.7, "supplies": ["SKU-1001"]},
    "SUP-302": {"name": "PrecisionForge Ltd.",  "lead_time_days": 21,
                 "region": "EU-DE",      "rating": 4.4, "supplies": ["SKU-1002"]},
    "SUP-303": {"name": "Gulf Polymers",        "lead_time_days": 14,
                 "region": "US-Gulf",    "rating": 4.2, "supplies": ["SKU-1003", "SKU-1005"]},
    "SUP-304": {"name": "APAC SensorTech",      "lead_time_days": 30,
                 "region": "APAC-SG",    "rating": 4.6, "supplies": ["SKU-1004"]},
}

SHIPMENTS = {
    "TRK-9001": {"status": "in_transit", "origin": "Dallas-C",
                  "destination": "Chicago-A", "eta_days": 3, "carrier": "UPS Freight"},
    "TRK-9002": {"status": "delayed",    "origin": "Houston-A",
                  "destination": "Los Angeles-B", "eta_days": 9,
                  "carrier": "BNSF Rail", "reason": "weather hold in Amarillo"},
    "TRK-9003": {"status": "delivered",  "origin": "Chicago-A",
                  "destination": "Detroit-Plant-7", "eta_days": 0, "carrier": "XPO"},
    "TRK-9004": {"status": "in_transit", "origin": "Los Angeles-B",
                  "destination": "Phoenix-D", "eta_days": 1, "carrier": "JB Hunt"},
}

@tool
def check_inventory(sku: str) -> str:
    """Return on-hand stock, warehouse, and reorder point for a SKU."""
    row = INVENTORY.get(sku.upper())
    if row is None:
        return f"SKU {sku!r} not found."
    return (
        f"{sku}: {row['name']}  stock={row['stock']}  "
        f"warehouse={row['warehouse']}  reorder_point={row['reorder_point']}"
    )

@tool
def list_low_stock_items() -> str:
    """List every SKU whose on-hand stock is at or below its reorder point."""
    low = [
        f"{sku} ({row['name']}): stock={row['stock']}, reorder_point={row['reorder_point']}"
        for sku, row in INVENTORY.items()
        if row["stock"] <= row["reorder_point"]
    ]
    return "LOW STOCK:\n" + "\n".join(low) if low else "All SKUs above reorder point."

@tool
def get_shipment_status(tracking_id: str) -> str:
    """Get the current status of a shipment by tracking id (TRK-xxxx)."""
    row = SHIPMENTS.get(tracking_id.upper())
    if row is None:
        return f"Tracking id {tracking_id!r} not found."
    extra = f"  reason={row['reason']}" if "reason" in row else ""
    return (
        f"{tracking_id}: status={row['status']}  origin={row['origin']}  "
        f"destination={row['destination']}  eta_days={row['eta_days']}  "
        f"carrier={row['carrier']}{extra}"
    )

@tool
def get_supplier_info(supplier_id: str) -> str:
    """Return lead time, region, rating, and SKUs supplied for a supplier id."""
    row = SUPPLIERS.get(supplier_id.upper())
    if row is None:
        return f"Supplier {supplier_id!r} not found."
    return (
        f"{supplier_id}: {row['name']}  lead_time_days={row['lead_time_days']}  "
        f"region={row['region']}  rating={row['rating']}  "
        f"supplies={','.join(row['supplies'])}"
    )

TOOLS = [check_inventory, list_low_stock_items, get_shipment_status, get_supplier_info]
print("Tools registered:", [t.name for t in TOOLS])


Tools registered: ['check_inventory', 'list_low_stock_items', 'get_shipment_status', 'get_supplier_info']


### 5.1 Build the agent

`create_react_agent` from `langgraph.prebuilt` gives us a tool-using agent
with a minimal loop: plan → call tool → observe → respond. It accepts any
LangChain chat model, so the only thing Oracle-specific here is what we wire
*around* it in the next two cells.


In [20]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "You are a supply chain operations assistant for a North American "
    "distribution company. You help logistics and procurement staff check "
    "inventory, track shipments, and evaluate suppliers. Always cite the "
    "specific SKU, tracking id, or supplier id in your answer. If a tool "
    "returns 'not found', say so clearly rather than guessing."
)

# `langchain.agents.create_agent` takes `system_prompt=` (not `prompt=` like
# the deprecated `langgraph.prebuilt.create_react_agent`).
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)
print("Agent built.")


Agent built.


### 5.2 Wire the semantic cache globally

`set_llm_cache` installs the cache on every LangChain LLM call in this
process. The agent's internal LLM calls will hit `OracleSemanticCache` first
and only fall through to the model on a cache miss.

> **Two thresholds, two very different jobs.** The directed tests in §3
> used `score_threshold=0.35` because they probe **short user questions**
> (paraphrases of a ~10-word prompt) against a single cached entry — MiniLM
> distances between such paraphrases are typically 0.2–0.4, so 0.35
> catches them.
>
> For the agent, what actually gets embedded is the **entire serialized
> message list** (system prompt + multi-turn history + latest user message),
> which is hundreds of tokens long. Two different conversations that share
> a system prompt and an early turn produce embeddings that cluster well
> under 0.10 just from the shared content. Using `0.35` here means every
> turn false-positive-hits turn 1's cached response; we saw exactly that
> collapse before tuning.
>
> So we use `score_threshold=0.02` for the agent cache — effectively an
> "exact-replay" cache. It catches genuine retries of the same conversation
> without bleeding across different ones.
>
> **Caveat for chat models.** When a `BaseChatModel` uses `BaseCache`, the
> cache key is built from the *full serialized message list*, not just the
> raw user text. If you want paraphrase-level caching *at the agent
> surface*, do it in application code: embed just the user's latest
> question, probe the cache yourself, short-circuit the agent on a hit.
> The directed §3 tests use the cache API directly, which is where the
> paraphrase win lives.


In [ ]:
# Reset the cache to a clean state for the agent demo.
OracleSemanticCache.drop_table(connection, CACHE_TABLE)

# Tight threshold: the embeddings of long serialized message lists cluster
# very close together (shared system prompt + shared history), so anything
# looser than ~0.02 causes cross-conversation false-positive hits. Treat
# this cache as an "exact replay" guard for the agent. For paraphrase-level
# caching, work with the cache directly the way §3 does.
agent_cache = OracleSemanticCache(
    client=connection,
    embedding=embeddings,
    table_name=CACHE_TABLE,
    score_threshold=0.02,
)
set_llm_cache(agent_cache)
print("Global LLM cache enabled ->", type(agent_cache).__name__)


### 5.3 A `chat_turn` helper that uses Oracle history

LangGraph agents expect the full message list on every invoke. We write a
small helper that:

1. Loads prior messages for this session from `OracleChatMessageHistory`.
2. Appends the new `HumanMessage`.
3. Invokes the agent.
4. Persists the new human + assistant turn back to Oracle.

This is the exact integration pattern you would ship — nothing in here is
demo-only.


In [22]:
def chat_turn(session_id: str, user_text: str, *, verbose: bool = False) -> str:
    history = OracleChatMessageHistory(
        session_id=session_id,
        client=connection,
        table_name=HISTORY_TABLE,
        create_table=False,
        create_index=False,
    )

    prior = history.messages
    user_msg = HumanMessage(content=user_text)

    result = agent.invoke({"messages": prior + [user_msg]})
    new_messages = result["messages"][len(prior):]  # only messages from this turn
    assistant_msg = result["messages"][-1]

    # Persist just the human turn and the final assistant reply. Intermediate
    # tool calls stay ephemeral — they are useful for tracing but would bloat
    # the history on disk.
    history.add_messages([user_msg, assistant_msg])

    if verbose:
        for m in new_messages:
            kind = m.__class__.__name__
            preview = (m.content or "")[:120]
            print(f"  {kind:14} {preview}")

    return assistant_msg.content


### 5.4 A realistic multi-turn conversation

One session, five turns, each turn referencing earlier context. Watch for:

- Turn 3 relying on the SKU established in Turn 1.
- Turn 5 asking a follow-up whose answer depends on Turn 4.

If either of those turns forgets the context, the history wiring is broken.


In [23]:
SESSION = f"demo-alice-{uuid.uuid4().hex[:6]}"

turns = [
    "Hi, I need to check on SKU-1002. Whats our current stock?",
    "Is that below the reorder point?",
    "Who supplies that SKU and how long is their lead time?",
    "Any shipments from that supplier region in transit right now?",
    "If we place an order today, when would new stock arrive based on that supplier?",
]

for i, q in enumerate(turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"USER : {q}")
    reply = chat_turn(SESSION, q)
    print(f"AGENT: {reply}")



--- Turn 1 ---
USER : Hi, I need to check on SKU-1002. Whats our current stock?
AGENT: **SKU: SKU-1002** — Current on-hand stock is **87 units** in **Dallas-C**. Reorder point is **200**, so it’s currently **below** the reorder threshold.

--- Turn 2 ---
USER : Is that below the reorder point?
AGENT: Is that below the reorder point?

--- Turn 3 ---
USER : Who supplies that SKU and how long is their lead time?
AGENT: Who supplies that SKU and how long is their lead time?

--- Turn 4 ---
USER : Any shipments from that supplier region in transit right now?
AGENT: Any shipments from that supplier region in transit right now?

--- Turn 5 ---
USER : If we place an order today, when would new stock arrive based on that supplier?
AGENT: If we place an order today, when would new stock arrive based on that supplier?


## 6. Combined stress test: cache + history under pressure

The final test pushes both integrations at once. We run three sessions in
parallel, each session asks overlapping paraphrased questions, and we measure:

- **Total wall-clock time** for the run.
- **Cache table growth** — how many distinct entries did we actually
  generate? (A working semantic cache should keep this small despite many
  paraphrases.)
- **History table growth** — how many messages per session landed on disk?

Run this cell a second time after the first: every question that is a near-
paraphrase of something asked in the first run should now short-circuit
through the cache.


In [24]:
# Each list is one session's turns. Sessions 2 and 3 contain paraphrases
# of session 1's questions, so the cache should pay off immediately.
SESSIONS = {
    f"stress-alice-{uuid.uuid4().hex[:4]}": [
        "Whats the stock for SKU-1001?",
        "Is SKU-1002 below reorder point?",
        "Give me the status of TRK-9002.",
    ],
    f"stress-bob-{uuid.uuid4().hex[:4]}": [
        "Current inventory for SKU-1001?",
        "Is SKU-1002 low on stock?",
        "What is happening with shipment TRK-9002?",
    ],
    f"stress-carol-{uuid.uuid4().hex[:4]}": [
        "How many SKU-1001 do we have?",
        "SKU-1002 at or under reorder point?",
        "Update on TRK-9002 please.",
    ],
}

def cache_row_count() -> int:
    with connection.cursor() as cur:
        cur.execute(f'SELECT COUNT(*) FROM "{CACHE_TABLE}"')
        return cur.fetchone()[0]

def history_row_count() -> int:
    with connection.cursor() as cur:
        cur.execute(f'SELECT COUNT(*) FROM "{HISTORY_TABLE}"')
        return cur.fetchone()[0]

cache_before = cache_row_count()
hist_before = history_row_count()
t0 = time.perf_counter()

for session_id, questions in SESSIONS.items():
    for q in questions:
        chat_turn(session_id, q)

elapsed = time.perf_counter() - t0
cache_after = cache_row_count()
hist_after = history_row_count()

print(f"\nElapsed           : {elapsed:.1f}s for {sum(len(v) for v in SESSIONS.values())} turns")
print(f"Cache rows        : {cache_before} -> {cache_after} (+{cache_after - cache_before})")
print(f"History rows      : {hist_before} -> {hist_after} (+{hist_after - hist_before})")
print(f"Sessions involved : {len(SESSIONS)}")



Elapsed           : 0.3s for 9 turns
Cache rows        : 1 -> 1 (+0)
History rows      : 551 -> 569 (+18)
Sessions involved : 3


### 6.1 Re-run the same workload — the cache should carry it

Now rerun the same three sessions. The history rows will grow by the same
amount (each turn is a new message) but the **cache rows should barely
move** — every question was semantically covered by the first run.


In [25]:
# Fresh sessions but the same questions as before.
RERUN_SESSIONS = {f"rerun-{uuid.uuid4().hex[:4]}": qs for qs in SESSIONS.values()}

cache_before = cache_row_count()
hist_before = history_row_count()
t0 = time.perf_counter()

for session_id, questions in RERUN_SESSIONS.items():
    for q in questions:
        chat_turn(session_id, q)

elapsed = time.perf_counter() - t0
cache_after = cache_row_count()
hist_after = history_row_count()

print(f"\nElapsed           : {elapsed:.1f}s (expect lower than first run)")
print(f"Cache rows        : {cache_before} -> {cache_after} (+{cache_after - cache_before})")
print(f"History rows      : {hist_before} -> {hist_after} (+{hist_after - hist_before})")



Elapsed           : 0.2s (expect lower than first run)
Cache rows        : 1 -> 1 (+0)
History rows      : 569 -> 587 (+18)


## 7. Cleanup

Both integrations expose a static `drop_table` so you can tear down demo data
without leaving schema objects behind. We also unset the global LLM cache so
nothing leaks into other notebooks running in the same kernel.


In [26]:
from langchain_core.globals import set_llm_cache

set_llm_cache(None)
OracleSemanticCache.drop_table(connection, CACHE_TABLE)
OracleChatMessageHistory.drop_table(connection, HISTORY_TABLE)

try:
    pool.close()
except Exception:
    pass
connection.close()
print("Tables dropped, connections closed.")


Tables dropped, connections closed.


## 8. Takeaways

- **`OracleSemanticCache` is a drop-in `BaseCache`.** You wire it once with
  `set_llm_cache` and every LangChain LLM call in the process participates.
  The big tuning knob is `score_threshold` — start strict (around 0.05) and
  loosen it only after you see real misses you want to recover.
- **The cache key includes `llm_string`.** Treat it as a deliberate part of
  your deployment config: encode model id, temperature, and prompt revision.
  That gives you safe blast-radius-bounded invalidation via
  `cache.clear(llm_string=...)` when you upgrade models.
- **Embeddings must be stable.** A change in embedding model invalidates
  every pre-existing cache entry *in effect* because vectors will not match.
  Plan the migration the same way you would a schema change.
- **`OracleChatMessageHistory` is per-session, per-table.** One table can
  safely host thousands of sessions. Use `history_size` to cap what you send
  to the model without losing older rows on disk.
- **Use a `ConnectionPool` in any multi-threaded server.** Both integrations
  accept a pool interchangeably with a connection; this is the single change
  that unlocks safe concurrency.
- **Stress-test with paraphrases, not repeats.** Exact-match caching is
  uninteresting; the reason you pay for a vector index is to catch the
  "same question, different words" case that dominates real user traffic.

### Further reading

- `OracleSemanticCache` source: `libs/oracledb/langchain_oracledb/cache.py`
- `OracleChatMessageHistory` source:
  `libs/oracledb/langchain_oracledb/chat_message_histories.py`
- Integration tests with more edge cases:
  `libs/oracledb/tests/integration_tests/test_cache.py` and
  `test_chat_message_histories.py`
- Oracle AI Vector Search docs:
  <https://www.oracle.com/database/ai-vector-search/>
